In [1]:
import os
import cv2
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 4
BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-4
CLIP_LEN = 16
IMG_SIZE = 112

train_dir = "/kaggle/input/datasets/yash07yadav/project-data/Complete Dataset/train"
val_dir   = "/kaggle/input/datasets/yash07yadav/project-data/Complete Dataset/val"

In [3]:
class ViolenceDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.classes = sorted(os.listdir(root_dir))

        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            for video in os.listdir(class_path):
                self.samples.append((os.path.join(class_path, video), label))

        self.label_map = {cls: idx for idx, cls in enumerate(self.classes)}

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        motion_scores = []

        prev_gray = None

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            if prev_gray is not None:
                diff = cv2.absdiff(prev_gray, gray)
                motion_score = np.mean(diff)
                motion_scores.append(motion_score)
                frames.append(frame)

            prev_gray = gray

        cap.release()

        if len(frames) == 0:
            frames = [np.zeros((IMG_SIZE, IMG_SIZE, 3))] * CLIP_LEN

        # Select top motion frames
        if len(motion_scores) > 0:
            idx = np.argsort(motion_scores)[-CLIP_LEN:]
            idx = sorted(idx)  # keep temporal order
            frames = [frames[i] for i in idx]

        # Pad if needed
        if len(frames) < CLIP_LEN:
            frames += [frames[-1]] * (CLIP_LEN - len(frames))

        frames = np.array(frames) / 255.0
        frames = np.transpose(frames, (3, 0, 1, 2))  # C,T,H,W

        return torch.tensor(frames, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        video = self.load_video(video_path)
        label = self.label_map[label]
        return video, label

In [4]:
train_dataset = ViolenceDataset(train_dir)
val_dataset   = ViolenceDataset(val_dir)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
from torchvision.models.video import r3d_18

model = r3d_18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:02<00:00, 53.0MB/s] 


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for videos, labels in tqdm(train_loader):
        videos = videos.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    print(f"Epoch {epoch+1}")
    print(f"Loss: {total_loss/len(train_loader):.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")

 56%|█████▌    | 363/647 [1:06:31<51:34, 10.90s/it]  

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for videos, labels in val_loader:
        videos = videos.to(DEVICE)
        outputs = model(videos)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Validation Accuracy:", accuracy_score(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": train_dataset.classes
}, "violence_r3d18_4class.pth")